Genera dei video annotati(boundingboxes + captioning)

In [ ]:
!{sys.executable} -m vista.annotate_video --video input.mp4 --out annotated.mp4 --config config/qwenyolo/mycfg.yaml --yolo-weights yolo12m.pt --start-frame 6000 --end-frame 7000   

Esegue MyPipeline la quale passa al VLM i crops di YOLO

In [ ]:
from vista.pipeline.mypipeline import build_mypipeline_from_config

pipeline = build_mypipeline_from_config(
    config_path="config/qwenyolo/mycfg.yaml",
    yolo_weights="yolo12m.pt",
)

from vista.run_submission import generate_submission
generate_submission(pipeline, "VistaVideos", "workspace/vista_output/MyPipelineSubmission.csv")

Il codice usato qui sotto è stato usato per valutare i detector(YOLO12m e YOLO11e-fine tuning), tuttavia il riscontro non è positivo per entrambi, YOLO12m sembrerebbe essere migliore ma entrambi non riescono ad identificare soggetti nei video INFRAred.

In [ ]:
COCO_CATEGORY_MAP = {
    "car": "car", "truck": "car", "bus": "car", "motorcycle": "car",
    "person": "person",
}
YOLOE_CATEGORY_MAP = {
    "crashed car": "car", "car": "car", "person": "person",
}

In [ ]:
from ultralytics import YOLO, YOLOE
from vista.qwen import get_model
import yaml

cfg  = yaml.safe_load(open("mycfg.yaml"))  #path of the mycfg 
qcfg = cfg.get("qwen", {})
captioner = QwenCropCaptioner(get_model(cfg), system_prompt=qcfg.get("system_prompt"))  # uno solo

def make_pipe(detector, category_map):
    return MyPipeline(detector, captioner, category_map=category_map,
                      caption_stride=qcfg.get("every_n_frames", 30),
                      crop_frac=0.1, min_crop_size=256)

coco_pipe  = make_pipe(YOLO("yolo12m.pt"),                                   COCO_CATEGORY_MAP)
yoloe_pipe = make_pipe(YOLOE("runs/detect/vistacrash_lp/weights/best.pt"),   YOLOE_CATEGORY_MAP)

In [ ]:
import cv2
from pathlib import Path
from PIL import Image

def run_on_video(pipeline, video_in, video_out):
    cap = cv2.VideoCapture(video_in)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Path(video_out).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    pipeline.reset()
    all_tracks, captioned = set(), set()
    idx = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        frame = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        res = pipeline.forward(frame, idx)
        for det in res.detections:
            all_tracks.add(det.track_id)
            if det.caption:
                captioned.add(det.track_id)
            x1, y1, x2, y2 = map(int, det.bbox)
            color = (0, 200, 0) if det.caption else (0, 0, 255)   # verde=captionato, rosso=no
            cv2.rectangle(bgr, (x1, y1), (x2, y2), color, 2)
            txt = f"{det.track_id} {det.category}: {det.caption or '...'}"
            cv2.putText(bgr, txt, (x1, max(12, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        writer.write(bgr)
        idx += 1
    cap.release(); writer.release()
    cov = len(captioned) / max(1, len(all_tracks))
    print(f"{Path(video_out).name}: {idx} frame | {len(all_tracks)} track | "
          f"coverage {cov:.1%} ({len(captioned)}/{len(all_tracks)})")
    return cov

In [ ]:
videos = ["data/sequences/bike_3.mp4", "data/sequences/era_collision.mp4"]  # i tuoi
for v in videos:
    name = Path(v).stem
    run_on_video(coco_pipe,  v, f"out/compare/{name}_coco.mp4")
    run_on_video(yoloe_pipe, v, f"out/compare/{name}_yoloe.mp4")